# Re-do 21 to check how much time do we need to solve the puzzle.

# Robot pushing

## Mapping 0

```txt
+---+---+---+
| 7 | 8 | 9 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
    | 0 | A |
    +---+---+
```

## Mapping 1

```txt
    +---+---+
    | ^ | A |
+---+---+---+
| < | v | > |
+---+---+---+
```

In [1]:
dic_numpad = {
    "7": (0, 0), "8": (0, 1), "9": (0, 2),
    "4": (1, 0), "5": (1, 1), "6": (1, 2),
    "1": (2, 0), "2": (2, 1), "3": (2, 2),
                 "0": (3, 1), "A": (3, 2)
}

dic_keypad = {
                 "^": (0, 1), "A": (0, 2),
    "<": (1, 0), "v": (1, 1), ">": (1, 2)
}

## Puzzle input

In [2]:
lst_puzzle = ["805A", "983A", "149A", "413A", "582A"]

## Transition computation

#### Compute transition for numpad

In [3]:
dic_trans_num = {}
for k0, (r0, c0) in dic_numpad.items():
    for k1, (r1, c1) in dic_numpad.items():
        dr = r1 - r0
        dc = c1 - c0
        pr = ("v" if dr > 0 else "^")  * abs(dr)
        pc = (">" if dc > 0 else "<")  * abs(dc)
        
        if dr == 0:
            dic_trans_num[k0+k1] = [pc + "A"]
        elif dc == 0:
            dic_trans_num[k0+k1] = [pr + "A"]
        elif (r0 == 3) & (c1 == 0): # Danger zone
            dic_trans_num[k0+k1] = [pr + pc + "A"]
        elif (r1 == 3) & (c0 == 0): # Danger zone
            dic_trans_num[k0+k1] = [pc + pr + "A"]
        else:
            dic_trans_num[k0+k1] = [pr + pc + "A", pc + pr + "A", ]
            
            

##### Compute transition for keypad

In [4]:
dic_trans_key = {}
for k0, (r0, c0) in dic_keypad.items():
    for k1, (r1, c1) in dic_keypad.items():
        dr = r1 - r0
        dc = c1 - c0
        pr = ("v" if dr > 0 else "^")  * abs(dr)
        pc = (">" if dc > 0 else "<")  * abs(dc)

        if dr == 0:
            dic_trans_key[k0+k1] = [pc + "A"]
        elif dc == 0:
            dic_trans_key[k0+k1] = [pr + "A"]
        elif (r0 == 0) & (c1 == 0): # Danger zone
            dic_trans_key[k0+k1] = [pr + pc + "A"]
        elif (r1 == 0) & (c0 == 0): # Danger zone 
            dic_trans_key[k0+k1] = [pc + pr + "A"]
        else:
            dic_trans_key[k0+k1] = [pc + pr + "A", pr + pc + "A"]


In [5]:
def filter_pattern_size(lst):
    """Filter path by length

    1. Search for the minimal path length
    2. Keep path only if they are minimal
    
    :param lst: list of path
    """
    l_min = min(map(len, lst))
    return list(filter(lambda x: len(x) == l_min, set(lst)))


In [6]:
def compute_string_numpad(pattern):
    """Compute possible paths leading to given pattern
    
    :param pattern: numeric code seqence (with numbers)
    :rparam: list of possible patterns (with dirpad)
    """
    full_pattern = "A" + pattern
    l = len(pattern)
    lst = [""]
    for i in range(l):
        pats = dic_trans_num[full_pattern[i:i+2]]
        lst_new = []
        for pat in pats:
            lst_new.extend([x + pat for x in lst])
        lst = lst_new

    return filter_pattern_size(lst)
        

In [7]:
def compute_string_keypad(pattern):
    """Compute possible paths leading to the provided pattern

    :param pattern: Direction sequence (with dirpad) (robot i)
    :rparam: list of possible patterns (with dirpad) (robot i+1)
    """
    full_pattern = "A" + pattern
    l = len(pattern)
    lst = [""]
    for i in range(l):
        pats = dic_trans_key[full_pattern[i:i+2]]
        lst_new = []
        for pat in pats:
            lst_new.extend([x + pat for x in lst])
        lst = lst_new

    return filter_pattern_size(lst)        

In [8]:
# Init solution counter
tot = 0 

# Loop over all inputs
for puzzle in lst_puzzle:
    c0 = int(puzzle[:-1])
    # Compute possible patterns for numpad
    lst = compute_string_numpad(puzzle)

    # Compute possible patterns for dirpad
    for _ in range(2):
        lst_next = []
        for seq in lst:
            ans = compute_string_keypad(seq)
            lst_next.extend(ans)

        # Keep only the shortest sequence
        lst = filter_pattern_size(lst_next)

    
    c1 = len(lst[0])
    # Display intermediate result
    print("{}: {} x {} = {}".format(puzzle, c1, c0, c0*c1))
    tot += c0*c1

# Print solution
print("----")
print(tot)

805A: 72 x 805 = 57960
983A: 66 x 983 = 64878
149A: 76 x 149 = 11324
413A: 70 x 413 = 28910
582A: 68 x 582 = 39576
----
202648


In [9]:
# Expected 202648

# Part 2

Recursively calculate best number of sequence.

Problem with part 1:

- The number of different actions a robot can do is very limited (5x5 possibilities). Useless to compute all
- Size of the full sequence is huge, something like `3**25` => Barrely fit in my computer.

Solution:

1. Memoization (store already explored path)
2. Do not store "full sequences", only subpath

In [10]:
n_lvl = 25

# List of dictionnary
# one list per level
# on each dic, pattern: len of the best path
lst_lvl = [{} for _ in range(n_lvl)]

# Init with keypad, so only dirpad in the recursion loop
lvl = 0
for key, lst in dic_trans_key.items():
    lst_lvl[0][key] = len(lst[0]) # At max 2 items in the list, each opti, so takes length of the first


def count_moves(pat, lvl):
    """Fx to compute pattern length
    """
    pattern = "A" + pat
    l = len(pattern)
    tot = 0
    for i in range(l-1):
        pp = pattern[i:i+2]
        score = lst_lvl[lvl-1][pp]
        tot += score
    
    return tot

for lvl in range(1, n_lvl):
    # Iteratively compute sequence length for all combination of all robots
    
    for key, lst in dic_trans_key.items():
        t0 = min(map(lambda x: count_moves(x, lvl), lst))
        lst_lvl[lvl][key] = t0    

In [11]:
lvl_x = 25-1 
# part 1: set to 2 -1
# part 2: set to 25 -1


lst_results = []
for puzzle in lst_puzzle:
    best = 10**100
    lst = compute_string_numpad(puzzle)
    for pat in lst:
        pattern = "A" + pat
        l = len(pattern)
        tot = 0
        for i in range(l-1):
            pp = pattern[i:i+2]
            tot += lst_lvl[lvl_x][pp]
        best = min(best, tot)
    
    print(puzzle, ":", best)
    lst_results.append(best * int(puzzle[:-1]))

805A : 86475783012
983A : 80732180764
149A : 91059074548
413A : 87288844796
582A : 86475783008


In [12]:
sum(lst_results)
#248919739734728

248919739734728